# **Synthetic DiD**
---

We will evaluate the effect of the first ceasefire agreements defined in 2 ways:

- <code>cea_agreement</code>: agreements in ceasefire stage 
- <code>ceasfire_agreements_mentions</code> agreements in ceasefire stage or mentioning ceasefire provisions

The use of *Synthetic DiD* in this case is usefull because we have **few treated units** and it has been seen the violation of paralell trends, and to construct a better counterfactual than simple averaging.


## **Outline**
---



In [33]:
import numpy as np
import pandas as pd
from diff_diff import SyntheticDiD, DifferenceInDifferences

# For nicer plots (optional)
try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed - visualization examples will be skipped")

# **Conflict Panel**
---

In [34]:
df = pd.read_csv('/Users/luciasauer/Library/CloudStorage/GoogleDrive-lucia.sauer@bse.eu/Mi unidad/EconAI/1_agreements_violence/agreements_violence/data/output/conflict_level/conflict_panel.csv')
#df_conflict = df[['isocode', 'year_mo', 'year_mo_numeric', 'log_best', 'cea_agreement', 'first_cea_agreement_date', 'treated_cea_agreement', 'ceasfire_agreements_mentions', 'treated_ceasfire_agreements_mentions']]
#df_conflict
df.columns

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_72445/3508638082.py:1: DtypeWarning: Columns (84,85,87,88,91,92,93,96,97,98,99,100,101,102,108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/luciasauer/Library/CloudStorage/GoogleDrive-lucia.sauer@bse.eu/Mi unidad/EconAI/1_agreements_violence/agreements_violence/data/output/conflict_level/conflict_panel.csv')


Index(['conflict_id', 'year_mo', 'best', 'n_events', 'n_isocode',
       'isocode_array', 'countries_array', 'type_of_violence', 'isocode',
       'country',
       ...
       'treated_agreements_no_ceasefire',
       'agreement_no_ceasefire_mentions_agreement_6m',
       'comp_agreement_no_ceasefire', 'treated_comp_agreements_no_ceasefire',
       'comp_agreement_no_ceasefire_mentions_agreement_6m',
       'subs_agreement_no_ceasefire', 'treated_subs_agreements_no_ceasefire',
       'subs_agreement_no_ceasefire_mentions_agreement_6m',
       'fatalities_pre_agreement', 'high_intensity'],
      dtype='object', length=300)

In [28]:
df.agreement.sum()

np.int64(745)

In [35]:
#show types of agreements for the first agreement
#df[df['first_agreement_date']!=0]
df.loc[df['first_agreement_date']!=0].groupby('first_agreement_date').conflict_id.nunique()

first_agreement_date
15     3
16     1
19     1
20     2
22     1
28     1
29     1
31     2
34     1
38     1
40     2
41     2
42     2
43     1
44     1
45     1
46     2
50     1
58     1
60     1
61     1
62     1
66     1
69     2
73     1
74     1
78     1
88     1
95     1
97     1
103    1
112    1
114    1
115    1
124    2
131    1
133    2
152    1
165    1
180    1
203    1
212    1
233    1
254    1
259    1
270    1
272    1
275    1
280    1
292    1
294    1
296    1
301    1
309    1
325    1
348    1
389    1
399    1
403    1
407    1
Name: conflict_id, dtype: int64

In [32]:
df.groupby('treated_agreement')['conflict_id'].nunique()

treated_agreement
0    130
1     71
Name: conflict_id, dtype: int64

## **Country Panel**
---

In [2]:
df_country = pd.read_csv('/Users/luciasauer/Library/CloudStorage/GoogleDrive-lucia.sauer@bse.eu/Mi unidad/EconAI/1_agreements_violence/agreements_violence/data/output/country_level/country_panel.csv')
df_country = df_country[['isocode', 'year_mo', 'year_mo_numeric', 'log_best', 'cea_agreement', 'first_cea_agreement_date', 'treated_cea_agreement', 'ceasfire_agreements_mentions', 'treated_ceasfire_agreements_mentions']]
df_country

,isocode,year_mo,year_mo_numeric,log_best,cea_agreement,first_cea_agreement_date,treated_cea_agreement,ceasfire_agreements_mentions,treated_ceasfire_agreements_mentions
0,AFG,1989-01,1,6.543552,0,0,0.0,0,1
1,AFG,1989-02,2,5.098341,0,0,0.0,0,1
2,AFG,1989-03,3,7.465512,0,0,0.0,0,1
3,AFG,1989-04,4,6.208087,0,0,0.0,0,1
4,AFG,1989-05,5,6.091310,0,0,0.0,0,1
...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,428,0.000000,0,0,0.0,0,0
46652,ZAF,2024-09,429,0.000000,0,0,0.0,0,0
46653,ZAF,2024-10,430,0.000000,0,0,0.0,0,0
46654,ZAF,2024-11,431,0.000000,0,0,0.0,0,0


In [3]:
df_country

,isocode,year_mo,year_mo_numeric,log_best,cea_agreement,first_cea_agreement_date,treated_cea_agreement,ceasfire_agreements_mentions,treated_ceasfire_agreements_mentions
0,AFG,1989-01,1,6.543552,0,0,0.0,0,1
1,AFG,1989-02,2,5.098341,0,0,0.0,0,1
2,AFG,1989-03,3,7.465512,0,0,0.0,0,1
3,AFG,1989-04,4,6.208087,0,0,0.0,0,1
4,AFG,1989-05,5,6.091310,0,0,0.0,0,1
...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,428,0.000000,0,0,0.0,0,0
46652,ZAF,2024-09,429,0.000000,0,0,0.0,0,0
46653,ZAF,2024-10,430,0.000000,0,0,0.0,0,0
46654,ZAF,2024-11,431,0.000000,0,0,0.0,0,0


In [4]:
#how many treated/control units?
df_country.groupby('first_cea_agreement_date')['isocode'].nunique()

first_cea_agreement_date
0      65
15      1
20      1
24      1
29      1
38      1
40      1
42      1
43      1
52      1
54      1
55      1
59      1
62      1
66      1
68      1
69      1
73      1
74      2
78      1
88      1
103     1
116     1
124     1
125     1
127     1
131     1
132     1
138     2
157     1
166     1
168     1
210     1
273     1
280     1
287     1
289     1
297     1
305     1
309     1
316     1
402     1
Name: isocode, dtype: int64

In [6]:
df_country.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46656 entries, 0 to 46655
Data columns (total 9 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   isocode                               46656 non-null  object 
 1   year_mo                               46656 non-null  object 
 2   year_mo_numeric                       46656 non-null  int64  
 3   log_best                              46656 non-null  float64
 4   cea_agreement                         46656 non-null  int64  
 5   first_cea_agreement_date              46656 non-null  int64  
 6   treated_cea_agreement                 46656 non-null  float64
 7   ceasfire_agreements_mentions          46656 non-null  int64  
 8   treated_ceasfire_agreements_mentions  46656 non-null  int64  
dtypes: float64(2), int64(5), object(2)
memory usage: 3.2+ MB


In [7]:
#if first_cea_agreement_date is before 2000 then set to 0 the value
df_country.loc[(df_country['first_cea_agreement_date']!=0) &  (df_country['year_mo'].str.contains('2000-'))]

,isocode,year_mo,year_mo_numeric,log_best,cea_agreement,first_cea_agreement_date,treated_cea_agreement,ceasfire_agreements_mentions,treated_ceasfire_agreements_mentions
564,AGO,2000-01,133,3.925268,0,29,1.0,0,1
565,AGO,2000-02,134,4.071872,0,29,1.0,0,1
566,AGO,2000-03,135,4.631487,0,29,1.0,0,1
567,AGO,2000-04,136,3.925268,0,29,1.0,0,1
568,AGO,2000-05,137,4.019382,0,29,1.0,0,1
...,...,...,...,...,...,...,...,...,...
45931,YEM,2000-08,140,0.000000,0,66,1.0,0,1
45932,YEM,2000-09,141,0.000000,0,66,1.0,0,1
45933,YEM,2000-10,142,2.995732,0,66,1.0,0,1
45934,YEM,2000-11,143,0.000000,0,66,1.0,0,1
